In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
    [results_1l, results_2l,],
    ignore_index=True )
#results = 

In [3]:
results.to_excel("resultados.xlsx")

In [4]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,MSE_ZZx1_theta,R2_ZZx2_theta,MSE_ZZx2_theta,...,R2_LSG_1_theta,MSE_LSG_1_theta,R2_LSG_2_theta,MSE_LSG_2_theta,R2_ZZx1_inv_theta,MSE_ZZx1_inv_theta,R2_zzx2_inv2_theta,MSE_zzx2_inv2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed8456,[1],0.3,0.7,0.01,8456,0.773577,0.615901,-0.927511,0.469200,...,0.847450,0.580099,-4.804612,0.303198,0.079276,0.429876,-6.542502,0.071416,-31.296092,-0.212938
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed2756,[1],0.3,0.7,0.01,2756,0.775190,0.618033,-0.955189,0.469819,...,0.835860,0.583288,-4.793429,0.306353,0.084620,0.433040,-6.569622,0.064487,-32.376022,-0.232333
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed1706,[1],0.3,0.7,0.01,1706,0.781015,0.638620,-1.106965,0.464580,...,0.830885,0.590243,-5.263732,0.299796,0.067949,0.429625,-6.771078,0.053694,-33.286567,-0.249102
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed4038,[1],0.3,0.7,0.01,4038,0.773034,0.639746,-1.128242,0.464934,...,0.816496,0.592751,-5.170622,0.303641,0.067199,0.433347,-6.794978,0.048743,-34.084957,-0.263081
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed6351,[1],0.3,0.7,0.01,6351,0.780101,0.620156,-0.977085,0.470076,...,0.819204,0.585669,-4.831333,0.307881,0.089469,0.434426,-6.596837,0.058034,-33.305975,-0.249706
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3131,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed4210,"[3, 1, 1]",0.7,0.3,0.90,4210,0.694944,0.550201,0.873411,0.409087,...,-1.745326,0.459606,-2.823176,0.260548,-0.811864,0.304416,-4.945941,0.176841,-13.112480,0.065411
3132,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed355,"[3, 1, 1]",0.7,0.3,0.90,355,0.687366,0.568452,0.801849,0.408228,...,-1.765957,0.467335,-2.868495,0.246510,-1.616160,0.302802,-7.450456,0.157408,-13.273439,0.066849
3133,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed7095,"[3, 1, 1]",0.7,0.3,0.90,7095,0.382804,0.364941,0.513469,0.227537,...,-8.326737,0.218690,-1.000825,0.152668,-0.876119,0.210851,-2.505922,0.165410,-1.799239,0.068982
3134,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed3174,"[3, 1, 1]",0.7,0.3,0.90,3174,0.440684,0.374710,0.545377,0.263651,...,-6.684607,0.271673,-1.615794,0.172718,-0.960664,0.188362,-2.858305,0.168093,-3.303254,0.122571


In [5]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv2": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] -
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
3134,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed3174,"[3, 1, 1]",0.440684,0.545377,-3.588375,-1.244932
3025,model_arch3-1_r0.9_Ld0.5_Lp0.5_seed7850,"[3, 1]",0.489393,0.676961,-3.765000,-1.251927
3008,model_arch3-1_r0.9_Ld0.5_Lp0.5_seed8672,"[3, 1]",0.509440,0.668794,-3.852042,-1.281075
3133,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed7095,"[3, 1, 1]",0.382804,0.513469,-3.544044,-1.293921
3116,model_arch3-1-1_r0.01_Ld0.3_Lp0.7_seed4210,"[3, 1, 1]",0.460859,0.376277,-3.601059,-1.303474



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
3134,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed3174,"[3, 1, 1]",0.440684,0.545377,-0.595910,-0.854162,-11.104234,-6.684607,-1.615794,-0.960664,-3.303254,0.440684,0.545377,-3.588375,-1.244932
3025,model_arch3-1_r0.9_Ld0.5_Lp0.5_seed7850,"[3, 1]",0.489393,0.676961,-0.294763,-1.591245,-11.614405,-5.337319,-1.678817,-0.874598,-4.963853,0.489393,0.676961,-3.765000,-1.251927
3008,model_arch3-1_r0.9_Ld0.5_Lp0.5_seed8672,"[3, 1]",0.509440,0.668794,-0.147812,-2.025653,-11.703889,-5.065595,-1.716489,-0.832265,-5.472589,0.509440,0.668794,-3.852042,-1.281075
3133,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed7095,"[3, 1, 1]",0.382804,0.513469,-1.024618,-0.433157,-11.347612,-8.326737,-1.000825,-0.876119,-1.799239,0.382804,0.513469,-3.544044,-1.293921
3116,model_arch3-1-1_r0.01_Ld0.3_Lp0.7_seed4210,"[3, 1, 1]",0.460859,0.376277,-0.210741,-0.748446,-11.293360,-6.569113,-1.243053,-1.543694,-3.599003,0.460859,0.376277,-3.601059,-1.303474


In [6]:
final_table.to_excel("BestModels.xlsx")

In [7]:
# ============================================
# MÉDIA, DESVIO, MÍNIMO E MÁXIMO
# ============================================

summary_tables = {}

for target in TARGETS:

    top_df = results.copy()
    rows = []

    for s in SETS_CATEGORY.keys():

        r2_col = f"R2_{s.replace('-', '_')}_{target}"
        mse_col = f"MSE_{s.replace('-', '_')}_{target}"

        row = {
            "Set": s,
            "Category": SETS_CATEGORY[s]
        }

        # =========================
        # R²
        # =========================
        if r2_col in top_df.columns:
            row["R2_mean"] = top_df[r2_col].mean()
            row["R2_std"]  = top_df[r2_col].std()
            row["R2_min"]  = top_df[r2_col].min()
            row["R2_max"]  = top_df[r2_col].max()
        else:
            row["R2_mean"] = np.nan
            row["R2_std"]  = np.nan
            row["R2_min"]  = np.nan
            row["R2_max"]  = np.nan

        # =========================
        # MSE
        # =========================
        if mse_col in top_df.columns:
            row["MSE_mean"] = top_df[mse_col].mean()
            row["MSE_std"]  = top_df[mse_col].std()
            row["MSE_min"]  = top_df[mse_col].min()
            row["MSE_max"]  = top_df[mse_col].max()
        else:
            row["MSE_mean"] = np.nan
            row["MSE_std"]  = np.nan
            row["MSE_min"]  = np.nan
            row["MSE_max"]  = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    summary_tables[target] = summary_df

    # =========================
    # MOSTRA SOMENTE A TABELA
    # =========================
    display(
        summary_df.style.format({
            "R2_mean": "{:.4f}",
            "R2_std":  "{:.4f}",
            "R2_min":  "{:.4f}",
            "R2_max":  "{:.4f}",

            "MSE_mean": "{:.6f}",
            "MSE_std":  "{:.6f}",
            "MSE_min":  "{:.6f}",
            "MSE_max":  "{:.6f}"
        })
    )

AttributeError: The '.style' accessor requires jinja2